# E1, E2, E3 — LLMs de Amazon Bedrock

## Preguntas
- **E1:** ¿un consenso de LLMs sin entrenar clasifica tan bien como el BERT afinado?
- **E2:** ¿cómo conviene combinar los juicios de varios LLMs en una escala ordinal?
- **E3:** ¿podemos ampliar el gold set (581) con auto-etiquetas de LLM sin meter ruido?

## Método
4 LLMs de Bedrock (Nova Micro/Lite/Pro + Llama 3 8B), Converse API, temperature=0, prompt few-shot con la escala como aprobación política. Se comparan contra el humano con QWK/MAE y **bootstrap pareado**. Auto-etiquetas solo donde varios LLMs concuerdan (gate de kappa).

> Reproducibilidad: modelos con versión fija (us-east-1), respuestas crudas guardadas en `resultados/bedrock_raw.jsonl`. Nota honesta: temperature=0 NO garantiza determinismo perfecto entre versiones del modelo.
> Para RE-EJECUTAR: `python experimentos/src/classify_581.py` y `classify_pool.py` (requiere Bedrock).

In [ ]:
# --- Configuracion comun ---
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np, pandas as pd
import common as C
R = C.RESULTS
def load(f): return json.load(open(R / f))

# Patron de dos niveles: por defecto CARGA resultados ya calculados (segundos, sin GPU).
# Para RE-EJECUTAR desde cero (requiere GPU/Bedrock), pon RECOMPUTE=True.
RECOMPUTE = False

## E1 — Triple comparación (consenso-LLM vs humano vs BERT)

In [ ]:
e1 = load("e1_triple.json"); d = e1["delta_qwk_llm_minus_bert"]
pd.DataFrame([["BERT+LoRA (afinado)", round(e1["bert"]["qwk"],3), round(e1["bert"]["mae"],3)],
              ["Consenso-LLM (sin entrenar)", round(e1["llm_consensus"]["qwk"],3), round(e1["llm_consensus"]["mae"],3)]],
             columns=["clasificador","QWK","MAE"])

In [ ]:
print(f"Delta QWK (LLM - BERT) = {d['mean']:+.3f}  IC95 [{d['ci_low']:+.3f}, {d['ci_high']:+.3f}]  P(LLM>BERT) = {d['p_llm_gt_bert']:.2f}")
print("Interpretacion HONESTA: es un EMPATE estadistico (el IC roza 0).")
print("Marco correcto: con N=581, el fine-tuning NO logra superar a un LLM zero-shot.")

## E2 — Agregación ordinal

In [ ]:
e2 = load("e2_aggregation.json")
pd.DataFrame([[k, round(v["qwk"],3)] for k,v in e2["aggregations"].items()], columns=["agregacion","QWK"]).sort_values("QWK", ascending=False)

La **mediana** gana (respeta el orden y es robusta al LLM débil). Acuerdo inter-LLM (los 4 coinciden): solo 17.6% — los comentarios son genuinamente ambiguos.

## E3 — Auto-etiquetado por concordancia

In [ ]:
g = load("e3_gate.json")
print("La concordancia predice la calidad (en los 581 con verdad humana):")
for t,v in g["gate_581"].items(): print(f"  >={t} LLMs coinciden -> QWK vs humano = {v['qwk_vs_human']:.3f}")
g3 = load("e3_gate3.json"); d = g3["delta_qwk_vs_p0"]
print(f"\nReentrenado con +{g3['n_auto']} auto-etiquetas: QWK={g3['metrics']['qwk']:.3f}, delta {d['mean']:+.3f} IC[{d['ci_low']:+.3f},{d['ci_high']:+.3f}] (dentro del ruido)")

## Veredicto y amenazas
E1: el LLM iguala al BERT (hallazgo interesante y honesto). E2: usar mediana, no mayoría. E3: prometedor pero dentro del ruido; **riesgo de propagación de sesgo** (las auto-etiquetas vienen del propio LLM) mitigado con el gate de concordancia, pero el 17.6% de acuerdo total sesga hacia comentarios fáciles.